# 06 – District Metrics

**Objective:** Compute basic per-district building summaries (count, total area, mean area) from the
master building table and save results to `outputs/tables/district_metrics.csv`.

**Prerequisite:** Run notebook 05 to generate `buildings_master.gpkg`.

In [ ]:
from __future__ import annotations

import geopandas as gpd
import pandas as pd
import yaml
from pathlib import Path

In [ ]:
# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
PROJECT_ROOT = Path.cwd().parent
CONFIG_PATH  = PROJECT_ROOT / "config" / "config.yaml"

with CONFIG_PATH.open() as fh:
    cfg = yaml.safe_load(fh)

db = cfg["database"]
BUILDINGS_MASTER  = PROJECT_ROOT / db["buildings_master_path"]
DISTRICTS_PATH    = PROJECT_ROOT / db["districts_path"]
METRICS_OUT       = PROJECT_ROOT / db["district_metrics_path"]
METRICS_OUT.parent.mkdir(parents=True, exist_ok=True)

print("Configuration loaded.")

## 1. Load master buildings table

In [ ]:
master = gpd.read_file(BUILDINGS_MASTER, layer="buildings_master")
print(f"Loaded {len(master):,} buildings")
master[["area_m2", "district_id", "district_name"]].head()

## 2. Aggregate to district level

In [ ]:
metrics = (
    master.dropna(subset=["district_id"])
    .groupby(["district_id", "district_name"], as_index=False)
    .agg(
        building_count=("area_m2", "count"),
        total_area_m2=("area_m2", "sum"),
        mean_area_m2=("area_m2", "mean"),
        median_area_m2=("area_m2", "median"),
    )
)
metrics["total_area_m2"]   = metrics["total_area_m2"].round(1)
metrics["mean_area_m2"]    = metrics["mean_area_m2"].round(1)
metrics["median_area_m2"]  = metrics["median_area_m2"].round(1)

metrics.sort_values("building_count", ascending=False)

## 3. Join district area for density

In [ ]:
districts = gpd.read_file(DISTRICTS_PATH, layer="gdynia_districts")
districts["district_area_m2"] = districts.geometry.area.round(1)

metrics = metrics.merge(
    districts[["district_id", "district_area_m2"]],
    on="district_id",
    how="left",
)
metrics["building_density_per_km2"] = (
    (metrics["building_count"] / metrics["district_area_m2"] * 1e6).round(2)
)
metrics

## 4. Save results

In [ ]:
metrics.to_csv(METRICS_OUT, index=False)
print(f"Saved {len(metrics)} district rows to {METRICS_OUT}")